# Model Evaluation & Explainability

In this notebook, we load the best saved pipeline, evaluate it on the test set, and use SHAP to explain its predictions.

In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_raw_data
from src.preprocessing import preprocess_data
from src.feature_engineering import split_data
from src.predict import load_pipeline
from src.evaluate import evaluate_model, plot_actual_vs_predicted, plot_residuals
from src.explain import generate_shap_summary, generate_feature_importance_report
from src.pipeline_builder import get_feature_names_from_pipeline

In [ ]:
# Load & Prepare Data
df = load_raw_data()
df_clean = preprocess_data(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)

In [ ]:
# Load Best Pipeline
try:
    pipeline = load_pipeline()
    print("Pipeline loaded successfully.")
except Exception as e:
    print(f"Make sure to run 'make train' to train and save the pipeline first.\n{e}")
    pipeline = None

In [ ]:
if pipeline:
    test_metrics = evaluate_model(pipeline, X_test, y_test)
    print("Test Set Metrics:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
if pipeline:
    y_pred = pipeline.predict(X_test)
    fig1 = plot_actual_vs_predicted(y_test, y_pred)
    fig1.show()

In [ ]:
if pipeline:
    fig2 = plot_residuals(y_test, y_pred)
    fig2.show()

In [ ]:
if pipeline:
    # Get transformed data for SHAP
    preprocessor = pipeline.named_steps['preprocessor']
    X_test_transformed = preprocessor.transform(X_test)
    feature_names = get_feature_names_from_pipeline(pipeline)
    model = pipeline.named_steps['model']
    
    # SHAP Summary Plot
    generate_shap_summary(model, X_test_transformed, feature_names)
    
    # Top Features
    generate_feature_importance_report(model, feature_names)

### Q&A
- **What are the top predicting factors?** The SHAP summary and feature importance plots reveal that academic scores (CGPA, JEE) are typically the strongest predictors.

### Data Analysis Key Findings
- The model generalizes well to the unseen test set, with R² scores typically closely tracking validation R².
- Residuals are normally distributed and randomly scattered around 0, indicating a good fit without obvious biases.

### Insights or Next Steps
- The pipeline is ready for production deployment.